In [25]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf

pio.renderers.default = "notebook_connected"

# Lab 2: Regresja liniowa

Lab jest podzielony na dwie części:

1. **Implementacja** regresji liniowej od podstaw.
2. **Analiza** gotowej regresji liniowej z biblioteki `statsmodels` z rozbudowanymi raportami

## Część 1: Regresja liniowa od podstaw

W tym laboratorium opieramy się na materiałach z kursu CS229 (Stanford): https://cs229.stanford.edu/main_notes.pdf. Poniżej skrót najważniejszych funkcji, które nas interesują (notacja lekko odbiega od materiału źródłowego. Pozostawiłem natomiast wektor parametrów jako $\theta$, choć chyba częściej spotykany jest zapis $\beta$).

Model liniowy (zakłada obecność szumu gaussowskiego):

$$y^{(i)} = \theta^T x^{(i)} + \epsilon^{(i)}, \qquad \epsilon^{(i)} \sim \mathcal{N}(0, \sigma^2)$$

Ponieważ $\mathbb{E}[\epsilon] = 0$, jako predykcję przyjmujemy $h_\theta(x) = \mathbb{E}[y \mid x] = \theta^T x$. Minimalizujemy sumę kwadratów reszt:

Funkcja kosztu:

$$J(\theta) = \frac{1}{2} \|X\theta - y\|^2$$

Gradient:

$$\nabla_\theta J(\theta) = X^T(X\theta - y)$$

Batchowa aktualizacja metodą najmniejszych kwadratów:

$$\theta := \theta - \alpha \, X^T(X\theta - y)$$

Inkrementalna aktualizacja metodą najmniejszych kwadratów. Dla każdego $i$:

$$\theta := \theta - \alpha \left(h_\theta(x^{(i)}) - y^{(i)}\right) x^{(i)}$$

### Zadanie 1: Implementacja

Uzupełnij implementację poniższych funkcji, żeby otrzymać funkcjonalną regresję liniową w dwóch wariantach treningu (batch i iteracyjny).

In [26]:
MAX_ITERATION = 1000
LEARNING_RATE = 1e-5

def initialize_theta(size):
    # TODO: zwróć wektor wartości początkowych o rozmiarze `size`
    return np.zeros(size)

def loss_function(theta, x, y):
    # TODO: oblicz i zwróć wartość funkcji kosztu J(theta)
    return 0.5 * np.sum((x @ theta - y)**2)

def loss_gradient(theta, x, y):
    # TODO: oblicz i zwróć gradient J względem theta
    return x.T @ (x @ theta - y)

def batch_least_mean_squares(learning_rate, x, y):
    intercept = np.ones(y.size)
    x_with_intercept = np.column_stack((intercept, x))
    theta = initialize_theta(x_with_intercept.shape[1])
    iteration = 0

    while iteration < MAX_ITERATION:
        # TODO: zaktualizuj theta
        theta -= learning_rate * loss_gradient(theta, x_with_intercept, y)
        iteration += 1

    return theta, loss_function(theta, x_with_intercept, y)

def incremental_least_mean_squares(learning_rate, x, y):
    intercept = np.ones(y.size)
    x_with_intercept = np.column_stack((intercept, x))
    theta = initialize_theta(x_with_intercept.shape[1])
    iteration = 0

    while iteration < MAX_ITERATION:
        for i in range(y.size):
            # TODO: zaktualizuj theta
            x_i = x_with_intercept[i]
            y_i = y[i]

            error = x_i @ theta - y_i
            theta -= learning_rate * error * x_i

        iteration += 1

    return theta, loss_function(theta, x_with_intercept, y)

### Weryfikacja

Dla przypadku 1d oczekujemy $\theta_1 \approx 1$, $\theta_0 \approx 0$.

In [27]:
x_1d = np.linspace(0, 20).reshape(-1, 1)
y_1d = np.linspace(0, 20)

theta_batch_1d, loss_batch_1d = batch_least_mean_squares(LEARNING_RATE, x_1d, y_1d)
theta_inc_1d, loss_inc_1d = incremental_least_mean_squares(LEARNING_RATE, x_1d, y_1d)

print(f"Batch LMS:       theta={theta_batch_1d}, J={loss_batch_1d:.4f}")
print(f"Incremental LMS: theta={theta_inc_1d}, J={loss_inc_1d:.4f}")

Batch LMS:       theta=[0.06508101 0.99515902], J=0.0273
Incremental LMS: theta=[0.06551253 0.99516096], J=0.0276


### Zadanie 2: Analiza funkcji kosztu

Poniżej mamy dwa zestawy danych do wytrenowania modelu. Jest między nimi bardzo istotna różnica. Aby ją wyłapać, zwizualizuj najpierw powierzchnie funkcji kosztu w zależności od współczynników $\theta$ przy obu wymiarach $x$. $\theta$ przy wyrazie wolnym traktujemy jako ustalone (równe 0). Następnie skomentuj różnicę między dwoma wykresami (bez zastosowania skali logarytmicznej będą trudne do rozróżnienia). Z czego ona wynika?

In [28]:
x_2d_1 = np.linspace([0, 0], [20, 40], 50)
y_2d_1 = np.linspace(0, 20, 50)

theta_batch_2d_1, loss_batch_2d_1 = batch_least_mean_squares(LEARNING_RATE, x_2d_1, y_2d_1)
theta_inc_2d_1, loss_inc_2d_1 = incremental_least_mean_squares(LEARNING_RATE, x_2d_1, y_2d_1)

print(f"Batch LMS:       theta={theta_batch_2d_1}, J={loss_batch_2d_1:.4f}")
print(f"Incremental LMS: theta={theta_inc_2d_1}, J={loss_inc_2d_1:.4f}")

Batch LMS:       theta=[0.01304654 0.1998062  0.39961241], J=0.0011
Incremental LMS: theta=[0.01348554 0.19980647 0.39961294], J=0.0012


In [29]:
x_2d_2 = np.column_stack((np.linspace(0, 20, 50), np.random.uniform(0, 10, 50)))
y_2d_2 = np.linspace(0, 20, 50)

theta_batch_2d_2, loss_batch_2d_2 = batch_least_mean_squares(LEARNING_RATE, x_2d_2, y_2d_2)
theta_inc_2d_2, loss_inc_2d_2 = incremental_least_mean_squares(LEARNING_RATE, x_2d_2, y_2d_2)

print(f"Batch LMS:       theta={theta_batch_2d_2}, J={loss_batch_2d_2:.4f}")
print(f"Incremental LMS: theta={theta_inc_2d_2}, J={loss_inc_2d_2:.4f}")

Batch LMS:       theta=[ 0.03187911  0.9985257  -0.00227308], J=0.0033
Incremental LMS: theta=[ 0.0320742   0.99852031 -0.00226342], J=0.0034


In [30]:
# Tutaj umieść kod wizualizacji
def visualize(x, y, theta_batch, loss_batch, title):

    theta0 = 0
    theta1_vals = np.linspace(-5, 25, 100)
    theta2_vals = np.linspace(-5, 45, 100)
    Theta1, Theta2 = np.meshgrid(theta1_vals, theta2_vals)
    J = np.zeros_like(Theta1)

    for i in range(Theta1.shape[0]):
        for j in range(Theta1.shape[1]):
            theta = np.array([Theta1[i,j], Theta2[i,j]])  
            J[i,j] = loss_function(theta, x, y) 


    fig = go.Figure(data=[go.Surface(z=np.log(J), x=Theta1, y=Theta2)])
    fig.add_trace(go.Scatter3d(
    x=[theta_batch[0]], y=[theta_batch[1]], z=[loss_batch],
    mode='markers', marker=dict(size=5, color='red'), name='Batch LMS'
    ))
    fig.update_layout(title=title, 
                      scene=dict(xaxis_title='theta1', yaxis_title='theta2', zaxis_title='J'))

    return fig

In [31]:
fig1 = visualize(x_2d_1, y_2d_1, theta_batch_2d_1, loss_batch_2d_1, title='Loss function set 1')
fig1.show()

In [32]:
fig2 = visualize(x_2d_2, y_2d_2,theta_batch_2d_2, loss_batch_2d_2, title='Loss function set 2')
fig2.show()

A tutaj komentarz

Pierwszy wykres wygląda jak „złożona kartka papieru” z wieloma fałdami, co wynika z silnej korelacji między cechami danych i sprawia, że funkcja ma długie, łagodne minimum. Drugi wykres przypomina lej z wyraźnym minimum, ponieważ cechy są mniej skorelowane, co daje jednoznaczne globalne minimum. Różnica wynika z zależności między kolumnami danych: w pierwszym zestawie druga kolumna to dokładnie dwa razy pierwsza (korelacja liniowa), natomiast w drugim zestawie cechy są bardziej losowe i taki problem nie występuje

### Dane mieszkaniowe – Kraków

W danych znajduje się wstępnie obrobiony zbiór ofert sprzedaży mieszkań z Krakowa z kolumnami: `cena_brutto_pln`, `powierzchnia_uzytkowa`, `liczba_izb`, `kondygnacja`, `dzielnica`, który posłuży do kolejnych ćwiczeń.

In [33]:
housing_df = pd.read_csv(Path("../..") / "data" / "krakow_housing.csv", index_col=0)
housing_df = housing_df[housing_df['cena_brutto_pln'] < 20_000_000]
housing_df.info()

<class 'pandas.DataFrame'>
Index: 5736 entries, 0 to 27235
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   cena_brutto_pln        5736 non-null   float64
 1   powierzchnia_uzytkowa  5736 non-null   float64
 2   liczba_izb             5736 non-null   float64
 3   kondygnacja            5736 non-null   float64
 4   dzielnica              5735 non-null   float64
dtypes: float64(5)
memory usage: 268.9 KB


## Zadanie 3: Zależności nieliniowe
Regresję można bardzo łatwo zaadoptować do zmiennych nieliniowych. Dopasuj model regresji przewidujacy cenę brutto za pomocą zmiennych `powierzchnia_uzytkowa`, `liczba_izb`, `kondygnacja` oraz kwadratu `powierzchnia_uzytkowa`.

In [34]:
# Tutaj umieść kod

housing_df['powierzchnia_uzytkowa^2'] = housing_df['powierzchnia_uzytkowa'] ** 2

standard_scaler = StandardScaler()
X_scaled = standard_scaler.fit_transform(housing_df[['powierzchnia_uzytkowa', 'liczba_izb', 'kondygnacja', 'powierzchnia_uzytkowa^2']])
y = housing_df['cena_brutto_pln'].values

In [35]:
theta_batch_housing_df, loss_batch_housing_df = batch_least_mean_squares(LEARNING_RATE, X_scaled, y)

print(f"Batch LMS:       theta={theta_batch_housing_df}, J={loss_batch_housing_df:.4f}")

Batch LMS:       theta=[ 779013.68176953  385607.35560156 -144078.37261194   47469.82592231
    1709.70626087], J=480981098832214.1250


---

## Część 2: Regresja liniowa z biblioteką `statsmodels`

Biblioteka `statsmodels` dostarcza implementacje OLS z pełną diagnostyką statystyczną – odpowiednik `lm()` z R.

### Dane krakowskie – OLS

In [36]:
X_krk_simple = sm.add_constant(housing_df[['powierzchnia_uzytkowa']])
y_krk_ols = housing_df['cena_brutto_pln']

fit_krk_simple = sm.OLS(y_krk_ols, X_krk_simple).fit()
print(fit_krk_simple.summary())

                            OLS Regression Results                            
Dep. Variable:        cena_brutto_pln   R-squared:                       0.281
Model:                            OLS   Adj. R-squared:                  0.281
Method:                 Least Squares   F-statistic:                     2246.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        21:57:54   Log-Likelihood:                -82424.
No. Observations:                5736   AIC:                         1.649e+05
Df Residuals:                    5734   BIC:                         1.649e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                  7.826e+

### Wizualizacja z przedziałami ufności

In [37]:
x_grid_krk = np.linspace(
    housing_df['powierzchnia_uzytkowa'].min(),
    housing_df['powierzchnia_uzytkowa'].max(),
    200
)
X_grid_krk = sm.add_constant(x_grid_krk)
pred_krk = fit_krk_simple.get_prediction(X_grid_krk).summary_frame()

fig_krk_ols = px.scatter(
    housing_df.sample(min(2000, len(housing_df)), random_state=42),
    x='powierzchnia_uzytkowa', y='cena_brutto_pln',
    opacity=0.3,
    title='OLS: cena ~ powierzchnia (Kraków)',
    labels={'powierzchnia_uzytkowa': 'Powierzchnia [m²]', 'cena_brutto_pln': 'Cena brutto [PLN]'}
)
fig_krk_ols.add_trace(go.Scatter(x=x_grid_krk, y=pred_krk['mean'],
    mode='lines', name='OLS', line=dict(color='red', width=2)))
fig_krk_ols.add_trace(go.Scatter(x=x_grid_krk, y=pred_krk['mean_ci_upper'],
    mode='lines', name='CI górny', line=dict(color='red', dash='dash', width=1)))
fig_krk_ols.add_trace(go.Scatter(x=x_grid_krk, y=pred_krk['mean_ci_lower'],
    mode='lines', name='CI dolny', line=dict(color='red', dash='dash', width=1)))
fig_krk_ols.show()

### Regresja wielokrotna (Kraków)

In [38]:
fit_krk_multi = smf.ols(
    'cena_brutto_pln ~ powierzchnia_uzytkowa + liczba_izb + kondygnacja',
    data=housing_df
).fit()
print(fit_krk_multi.summary())

                            OLS Regression Results                            
Dep. Variable:        cena_brutto_pln   R-squared:                       0.321
Model:                            OLS   Adj. R-squared:                  0.320
Method:                 Least Squares   F-statistic:                     901.9
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        21:57:54   Log-Likelihood:                -82263.
No. Observations:                5736   AIC:                         1.645e+05
Df Residuals:                    5732   BIC:                         1.646e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept              7.064e+

### Dzielnica jako zmienna kategoryczna

`dzielnica` reprezentuje kategorię – używamy `C(dzielnica)` aby automatycznie wygenerować zmienne dummy. Model regresji przyjmuje wtedy następujacą formę:

$$\text{cena} = \theta_0 + \theta_1 \cdot \text{powierzchnia} + \theta_2 \cdot \text{liczba\_izb} + \theta_3 \cdot \text{kondygnacja} + \sum_{j=2}^{18} \gamma_j D_j + \epsilon$$

Pomijamy jedną kategorię, bo przy jej zawarciu dla parametrów zachodziłoby:

$$D_1 + D_2 + \cdots + D_{18} = 1 \quad \Rightarrow \quad D_1 = 1 - D_2 - \cdots - D_{18}$$

Czyli mielibyśmy powtórkę z przykładu ze współliniowością i nieskończenie wiele poprawnych zestawów parametrów.

In [39]:
fit_krk_cat = smf.ols(
    'cena_brutto_pln ~ powierzchnia_uzytkowa + liczba_izb + kondygnacja + C(dzielnica)',
    data=housing_df
).fit()
print(fit_krk_cat.summary())

                            OLS Regression Results                            
Dep. Variable:        cena_brutto_pln   R-squared:                       0.378
Model:                            OLS   Adj. R-squared:                  0.376
Method:                 Least Squares   F-statistic:                     173.8
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        21:57:54   Log-Likelihood:                -81996.
No. Observations:                5735   AIC:                         1.640e+05
Df Residuals:                    5714   BIC:                         1.642e+05
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept              2.199e+

## Ćwiczenie 1.
Zinterpretuj model. Co oznaczają kolejne elementy raportu? Które zmienne są najważniejsze w predykcji?

Tutaj zapisz wnioski

Model regresji liniowej pokazuje, jak różne cechy mieszkań wpływają na cenę brutto. Współczynniki (coef) informują, że największy wpływ mają powierzchnia użytkowa (im większe mieszkanie, tym wyższa cena) oraz dzielnica, w której się znajduje — niektóre dzielnice znacząco podnoszą lub obniżają cenę w porównaniu do dzielnicy bazowej. Kondygnacja i liczba izb mają umiarkowany wpływ, a większość zmiennych jest istotna statystycznie (P < 0.05). R² = 0.378 pokazuje, że model wyjaśnia około 38% zmienności cen, natomiast reszty są mocno skośne, co sugeruje, że prosta regresja liniowa nie uchwyciła wszystkich ekstremalnych wartości.

### Porównanie modeli (Kraków)

In [40]:
print(f"{'Model':<45} {'R²':>8}  {'AIC':>12}")
print("-" * 68)
for _name, _fit in [
    ("prosta: powierzchnia", fit_krk_simple),
    ("wielokrotna: pow + izby + piętro", fit_krk_multi),
    ("wielokrotna + C(dzielnica)", fit_krk_cat),
]:
    print(f"{_name:<45} {_fit.rsquared:>8.4f}  {_fit.aic:>12.1f}")

Model                                               R²           AIC
--------------------------------------------------------------------
prosta: powierzchnia                            0.2814      164852.7
wielokrotna: pow + izby + piętro                0.3207      164534.7
wielokrotna + C(dzielnica)                      0.3782      164033.6


### Wykres rezyduów

In [41]:
_fitted = fit_krk_cat.fittedvalues
_resid = fit_krk_cat.resid

fig_resid_krk = px.scatter(
    x=_fitted, y=_resid,
    title='Residua vs. Wartości dopasowane (Kraków)',
    labels={'x': 'Wartości dopasowane', 'y': 'Residua'}
)
fig_resid_krk.add_hline(y=0, line_dash="dash", line_color="red")
fig_resid_krk.show()

### Standaryzacja cech a gradient descent

Przy dużej różnicy skali między cechami krok uczenia musi być ekstremalnie mały.
Poniżej widać, że na surowych danych krakowskich potrzebujemy `LR = 1e-12`, żeby gradient descent w ogóle zbiegał (do wywołania komórek trzeba zaimplementować regresję):

In [42]:
x_krk_raw = housing_df[['powierzchnia_uzytkowa']].values
y_krk_raw = housing_df['cena_brutto_pln'].values

_, loss_div = batch_least_mean_squares(LEARNING_RATE, x_krk_raw, y_krk_raw)
print(f"LR={LEARNING_RATE} (surowe dane): J={loss_div:.2e}  ← dywergencja (nan/inf)")

LEARNING_RATE_KRK = 1e-12
theta_krk_raw, loss_krk_raw = batch_least_mean_squares(LEARNING_RATE_KRK, x_krk_raw, y_krk_raw)
print(f"LR={LEARNING_RATE_KRK}:           J={loss_krk_raw:.2e}, theta={np.round(theta_krk_raw, 0)}")

LR=1e-05 (surowe dane): J=nan  ← dywergencja (nan/inf)
LR=1e-12:           J=2.38e+15, theta=[  4. 270.]


C:\Users\Dominik\AppData\Local\Temp\ipykernel_16612\2043611719.py:14: RuntimeWarning: overflow encountered in matmul
  return x.T @ (x @ theta - y)
C:\Users\Dominik\AppData\Local\Temp\ipykernel_16612\2043611719.py:24: RuntimeWarning: invalid value encountered in subtract
  theta -= learning_rate * loss_gradient(theta, x_with_intercept, y)


Standaryzacja $x \leftarrow \frac{x - \mu}{\sigma}$ wyrównuje skale i pozwala używać standardowego `LEARNING_RATE`.
Dodatkowo uspójnia nam skalę współczynników między cechami, co mówi bardziej bezpośrednio o ich istotności.

In [43]:
scaler = StandardScaler()
x_krk_scaled = scaler.fit_transform(housing_df[['powierzchnia_uzytkowa']])
y_krk_scaled = housing_df['cena_brutto_pln'].values

theta_krk_scaled_batch, loss_krk_scaled_batch = batch_least_mean_squares(LEARNING_RATE, x_krk_scaled, y_krk_scaled)
theta_krk_scaled_inc, loss_krk_scaled_inc = incremental_least_mean_squares(LEARNING_RATE, x_krk_scaled, y_krk_scaled)

print(f"Batch LMS (skalowane):       theta={np.round(theta_krk_scaled_batch, 2)}, J={loss_krk_scaled_batch:.2e}")
print(f"Incremental LMS (skalowane): theta={np.round(theta_krk_scaled_inc, 2)}, J={loss_krk_scaled_inc:.2e}")

Batch LMS (skalowane):       theta=[779013.68 263563.57], J=5.09e+14
Incremental LMS (skalowane): theta=[778730.74 263784.57], J=5.09e+14


In [44]:
_housing_std = housing_df.copy()
_housing_std[['powierzchnia_uzytkowa', 'liczba_izb', 'kondygnacja']] = StandardScaler().fit_transform(
    housing_df[['powierzchnia_uzytkowa', 'liczba_izb', 'kondygnacja']]
)
fit_krk_std = smf.ols(
    'cena_brutto_pln ~ powierzchnia_uzytkowa + liczba_izb + kondygnacja + C(dzielnica)',
    data=_housing_std
).fit()
print(fit_krk_std.summary())

                            OLS Regression Results                            
Dep. Variable:        cena_brutto_pln   R-squared:                       0.378
Model:                            OLS   Adj. R-squared:                  0.376
Method:                 Least Squares   F-statistic:                     173.8
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        21:58:18   Log-Likelihood:                -81996.
No. Observations:                5735   AIC:                         1.640e+05
Df Residuals:                    5714   BIC:                         1.642e+05
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept              9.598e+

## Ćwiczenie 2.
Jak zmieniły się współczynniki przy zmiennych? Czy zmienia to poprzednia interpretację ich istotności z ćwiczenia 1?

Tutaj zapisz wnioski

Po standaryzacji współczynniki dla powierzchni użytkowej, liczby izb i kondygnacji wzrosły liczbowo, ponieważ mierzą teraz wpływ zmiany o 1 odchylenie standardowe, a nie o jednostkę w oryginalnej skali. Kierunek wpływu i poziom istotności tych zmiennych pozostał taki sam, więc interpretacja modelu nie zmienia się, natomiast łatwiej porównać względny wpływ cech na cenę.

In [48]:
fig_list = [fig1, fig2, fig_krk_ols, fig_resid_krk]
for i, fig in enumerate(fig_list, start=1):
    pio.write_html(fig, Path("../..")/"lab_files"/"lab2"/f"lab2_{i}.html", include_plotlyjs='inline', full_html=True)